In [1]:
import sys
from pathlib import Path

for src_path in (Path.cwd() / "src", Path.cwd().parent / "src"):
    if src_path.exists():
        sys.path.append(str(src_path))
from data import generate_induction_data
import torch
import transformers as tr
import transformer_lens as tl


In [2]:
gpt2_tokenizer = tr.GPT2Tokenizer.from_pretrained("gpt2")


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32
gpt2 = tl.HookedTransformer.from_pretrained("gpt2", device=device, dtype=dtype)
gpt2.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
      (h

In [ ]:
prompt = "James and Mary entered the market; While James was examining the apples and oranges, Mary was looking at the bananas. James said to"
tensors = gpt2_tokenizer(prompt, return_tensors="pt").to(device)
block_ids = [0, 1, 2]

def all_blocks_filter(name):
    return name.startswith("blocks.") and name.endswith(".hook_resid_post") or name == "ln_final.hook_scale"

_, cache = gpt2.run_with_cache(
    tensors["input_ids"],
    names_filter=all_blocks_filter,
    device=device,
)

from importlib import reload

import core.logit_lens.utils as logit_lens_utils
import core.logit_lens.visualization as logit_lens_visualization

logit_lens_utils = reload(logit_lens_utils)
logit_lens_visualization = reload(logit_lens_visualization)

get_block_logits = logit_lens_utils.get_block_logits
display_argmax_tokens = logit_lens_visualization.display_argmax_tokens
display_top_tokens = logit_lens_visualization.display_top_tokens

block_logits, decoded_tokens = get_block_logits(
    gpt2,
    block_ids,
    cache,
    gpt2_tokenizer,
    return_decoded=True,
)

block_logits.shape, decoded_tokens[0][0][-5:]

In [5]:
display_argmax_tokens(
    gpt2,
    block_logits,
    block_ids,
    gpt2_tokenizer,
    input_ids=tensors["input_ids"],
    positions=range(tensors["input_ids"].shape[1] - 8, tensors["input_ids"].shape[1]),
)

,18: looking,19: at,20: the,21: bananas,22: .,23: James,24: said,25: to
block,,,,,,,,
0,looking,least,same,bananas,\n,James,the,be
1,at,least,same,bananas,It,James,that,be
2,at,the,same,bananas,They,'s,he,be


,18: looking,19: at,20: the,21: bananas,22: .,23: James,24: said,25: to
block,,,,,,,,
0,looking,least,same,bananas,\n,James,the,be
1,at,least,same,bananas,It,James,that,be
2,at,the,same,bananas,They,'s,he,be


In [6]:
display_top_tokens(
    gpt2,
    block_logits,
    block_ids,
    gpt2_tokenizer,
    input_ids=tensors["input_ids"],
    positions=range(tensors["input_ids"].shape[1] - 5, tensors["input_ids"].shape[1]),
    top_k=5,
)

block,pos,input,rank,token,token_id,logit
0,21,bananas,1,bananas,35484,27.431
0,21,bananas,2,banana,25996,20.874
0,21,bananas,3,fruit,8234,17.891
0,21,bananas,4,toast,27805,16.972
0,21,bananas,5,Blossom,35544,15.839
0,22,.,1,\n,198,17.604
0,22,.,2,And,843,17.360
0,22,.,3,It,632,17.329
0,22,.,4,The,383,16.879
0,22,.,5,But,887,16.681


,block,pos,input,rank,token,token_id,logit
0,0,21,bananas,1,bananas,35484,27.431238
1,0,21,bananas,2,banana,25996,20.874313
2,0,21,bananas,3,fruit,8234,17.890532
3,0,21,bananas,4,toast,27805,16.971897
4,0,21,bananas,5,Blossom,35544,15.838716
...,...,...,...,...,...,...,...
70,2,25,to,1,be,307,19.800716
71,2,25,to,2,give,1577,19.365366
72,2,25,to,3,make,787,18.269447
73,2,25,to,4,get,651,17.984037
